# Chapter 2 — VectorRAG

> Embedding · Chunking · Vector DB · Hybrid Search · Reranking
> v2.0 / 2026 · NOWAVE

## 튜토리얼 구성

| § | 튜토리얼 | 도구 | 학습 포인트 |
|---|---|---|---|
| 2-1 | 60줄 최소 VectorRAG | OpenAI + Chroma + LangChain LCEL | 베이스라인 구현 |
| 2-2 | 임베딩 A/B 테스트 | text-embedding-3 small/large + truncate | 차원·정확도·비용 |
| 2-3 | 청킹 7종 비교 | Recursive·Semantic·Markdown·Late·Parent-Child·Contextual | RAGAS context_precision |
| 2-4 | 프로덕션 파이프라인 | Qdrant + BM25 + RRF + Cohere Rerank v3.5 | Hybrid·Rerank 효과 |
| 2-5 | 실패 모드 5가지 재현 | 다중 엔티티·표 셀·교차 참조·시점·반의어 | Ch.3~6의 존재 이유 |

## 0. 환경 준비

```bash
pip install -q openai langchain langchain-openai langchain-community \
   chromadb qdrant-client rank_bm25 cohere ragas pymupdf4llm \
   sentence-transformers tiktoken python-dotenv
```

본 노트북 전체 OpenAI 비용은 약 $3~$8 수준이다.

In [1]:
%pip install -q langchain langchain-openai langchain-community chromadb qdrant-client rank_bm25 cohere ragas pymupdf4llm sentence-transformers tiktoken

### 0.1 환경 변수 로드와 PDF 확인

Chapter 1에서 만든 `work/sample_10k.pdf`가 있어야 한다. 없으면 Chapter 1 노트북을 먼저 실행한다.

In [2]:
# ─────────────────────────────────────────────────────
# 환경 변수 로드 + Chapter 1 산출물 확인
# ─────────────────────────────────────────────────────
import os, json, time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# OPENAI_API_KEY 필수 (Cohere/LlamaParse는 선택)
#os.environ.setdefault("OPENAI_API_KEY", "sk-...")
# os.environ.setdefault("COHERE_API_KEY", "...")   # §2-4 선택

WORK = Path("./work"); WORK.mkdir(exist_ok=True)
PDF_PATH = WORK / "sample_10k.pdf"   # Chapter 1에서 생성

assert PDF_PATH.exists(), (
    "Chapter 1 노트북을 먼저 실행하여 sample_10k.pdf를 만드시오."
)
print("입력 PDF:", PDF_PATH)
print(f"크기: {PDF_PATH.stat().st_size:,} bytes")

입력 PDF: work/sample_10k.pdf
크기: 2,064 bytes


### 0.2 Chapter 1의 마크다운 산출물 재사용

본 챕터의 모든 임베딩·검색은 PyMuPDF4LLM이 추출한 마크다운 텍스트 위에서 동작한다. Chapter 1 §1-1에서 다룬 도구를 그대로 호출한다.

In [3]:
# ─────────────────────────────────────────────────────
# PyMuPDF4LLM으로 마크다운 추출 (Ch.1 §1-1 재사용)
# ─────────────────────────────────────────────────────
import pymupdf4llm

# page_chunks=True로 페이지별 분할 — 후속 청킹·검색에서 페이지 메타 활용
md_pages = pymupdf4llm.to_markdown(str(PDF_PATH), page_chunks=True)
full_text = "\n\n".join(p["text"] for p in md_pages)

print(f"페이지 {len(md_pages)}개, 총 {len(full_text):,}자")
print("\n첫 페이지 미리보기 (300자):")
print(full_text[:300])

=== Document parser messages ===
Using RapidOCR for OCR processing.

페이지 1개, 총 313자

첫 페이지 미리보기 (300자):
## **FY2025 Annual Report — Sample Corp** 

## **Chapter 1. Revenue** 

## **1.1 Q1 Results** 

|**Segment**|**Q1 Revenue (M$)**|**YoY %**|
|---|---|---|
|iPhone|69,702|+5.5%|
|Mac|7,744|+1.6%|
|Services|26,375|+11.5%|



## **Chapter 2. Risk Factors** 

Supply chain disruption is the primary risk i


---
## §2-1 60줄 최소 VectorRAG — OpenAI + Chroma + LangChain LCEL

가장 작은 작동 RAG를 만들어 본다. 이 60줄이 본 챕터의 베이스라인이며, 이후 §2-2~§2-4에서 단계적으로 강화한다.

**파이프라인 6단계**:
1. Recursive 청킹 (512 토큰 / 64 overlap)
2. text-embedding-3-small 임베딩
3. Chroma in-memory 저장
4. Top-k=5 유사도 검색
5. 프롬프트 + GPT-4o-mini 생성
6. LCEL 체인으로 묶기

In [4]:
# ─────────────────────────────────────────────────────
# §2-1: 60줄 최소 VectorRAG 베이스라인
# ─────────────────────────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Step 1 — 청킹: Recursive 512 토큰 / 64 오버랩
# (\n\n → \n → 공백 순으로 의미 단위를 우선 보존)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=64,
    separators=["\n\n", "\n", " ", ""],
)
chunks = splitter.split_text(full_text)
print(f"청크 수: {len(chunks)}")

# Step 2 + 3 — 임베딩 + 저장: text-embedding-3-small (1536d) + Chroma
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(
    chunks, embedding=embedding,
    collection_name="ch02_baseline",
)

# Step 4 — Retrieval: Top-k=5
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Step 5 — 프롬프트 + LLM
# (근거가 없으면 "문서에 명시되지 않았다"고 답하도록 명시 → hallucination 차단)
prompt = ChatPromptTemplate.from_template("""\
당신은 컨텍스트만 근거로 답하는 어시스턴트다.
근거가 없으면 '문서에 명시되지 않았다'고 답한다.

# 컨텍스트
{context}

# 질문
{question}

# 답변
""")
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)

# Step 6 — LCEL 체인 (LangChain Expression Language)
# context (retriever) + question (passthrough) → prompt → LLM → string
rag = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

# 첫 질의
answer = rag.invoke("FY2025 Q1 iPhone 매출은 얼마인가?")
print("=== 답변 ===")
print(answer)

/opt/miniconda3/envs/lecture/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


청크 수: 1


Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1


=== 답변 ===
FY2025 Q1 iPhone 매출은 **69,702 M$**입니다.


**관찰**: 60줄로 작동하는 RAG가 만들어졌다. 그러나 이 베이스라인은 §2-5의 5가지 실패 모드에서 무너진다. 이제 단계적으로 강화한다.

---
## §2-2 임베딩 A/B 테스트 — small vs large vs truncate

OpenAI의 text-embedding-3 시리즈는 **차원 truncate**를 지원한다. 1536d 모델을 512d로 잘라 써도 성능 손실이 1~2점에 그치므로 비용·저장 공간이 절감된다. 동일 쿼리에 대해 5개 조합을 측정한다.

In [5]:
# ─────────────────────────────────────────────────────
# §2-2 준비: OpenAI 임베딩 직접 호출 함수
# ─────────────────────────────────────────────────────
import numpy as np
from openai import OpenAI

client = OpenAI()

def embed(texts, model="text-embedding-3-small", dim=None):
    """OpenAI 임베딩 호출 — dim 옵션으로 차원 truncate"""
    kwargs = {"model": model, "input": texts}
    if dim:
        kwargs["dimensions"] = dim   # ← 차원 자르기 (small=1536, large=3072 기본)
    r = client.embeddings.create(**kwargs)
    return np.array([e.embedding for e in r.data])

# 쿼리·정답 쌍 (실제 환경에선 100개 이상 권장)
queries = [
    "iPhone Q1 매출",
    "Mac 사업부 성장률",
    "Services 부문 매출",
    "공급망 리스크",
    "회계 연도 2025",
]
print("쿼리 수:", len(queries))

쿼리 수: 5


### 2-2-1. 5가지 조합 평가 — 모델 × 차원

`small/large` × `512d truncate / full dim`의 조합으로 평균 Top-1 코사인 유사도를 측정한다. 실전에서는 RAGAS 또는 hit@k가 더 정확하지만, 본 데모에서는 간이 신호로 충분하다.

In [6]:
# ─────────────────────────────────────────────────────
# 5개 조합의 Top-1 코사인 유사도 측정
# ─────────────────────────────────────────────────────
def evaluate(model, dim):
    """청크·쿼리 임베딩 후 평균 Top-1 유사도 측정"""
    chunk_vecs = embed(chunks[:50], model=model, dim=dim)
    q_vecs = embed(queries, model=model, dim=dim)
    # 정규화 후 cos 유사도
    chunk_n = chunk_vecs / np.linalg.norm(chunk_vecs, axis=1, keepdims=True)
    q_n     = q_vecs     / np.linalg.norm(q_vecs,     axis=1, keepdims=True)
    sims = q_n @ chunk_n.T
    # 각 쿼리별 Top-1 유사도의 평균
    return float(sims.max(axis=1).mean())

results = {}
for model, dim in [
    ("text-embedding-3-small", None),       # 1536d 기본
    ("text-embedding-3-small", 512),        # 512d로 자르기
    ("text-embedding-3-large", None),       # 3072d 기본
    ("text-embedding-3-large", 512),        # 512d
    ("text-embedding-3-large", 1024),       # 1024d
]:
    key = f"{model.replace('text-embedding-3-','')}-d{dim or 'full'}"
    t0 = time.time()
    score = evaluate(model, dim)
    elapsed = time.time() - t0
    results[key] = (score, elapsed)
    print(f"{key:20s}  top-1 sim={score:.4f}  {elapsed:.2f}s")

small-dfull           top-1 sim=0.3313  0.43s
small-d512            top-1 sim=0.3720  0.82s
large-dfull           top-1 sim=0.3167  0.87s
large-d512            top-1 sim=0.3727  0.28s
large-d1024           top-1 sim=0.3456  0.88s


In [7]:
# ─────────────────────────────────────────────────────
# 결과를 pandas 테이블로 정리
# ─────────────────────────────────────────────────────
import pandas as pd
df = pd.DataFrame(
    [(k, *v) for k, v in results.items()],
    columns=["임베딩 모델·차원", "Top-1 유사도", "처리 시간(s)"]
)
df

,임베딩 모델·차원,Top-1 유사도,처리 시간(s)
0,small-dfull,0.331266,0.426134
1,small-d512,0.372027,0.818824
2,large-dfull,0.316725,0.873913
3,large-d512,0.372708,0.279040
4,large-d1024,0.345606,0.875502


---
## §2-3 청킹 전략 7종 비교

Recursive·Markdown·Semantic·Late·Parent-Child·Contextual의 영향을 RAGAS의 `context_precision`으로 비교한다 (이론 그림 2.3 참조).

### 2-3-1. 기본 3종 (Fixed / Recursive / Markdown)

가장 단순한 세 전략부터 시작한다. Fixed는 의미를 무시한 고정 분할, Recursive는 구분자 우선, Markdown-aware는 # 헤딩을 청크 경계로 사용한다.

In [8]:
# ─────────────────────────────────────────────────────
# 청킹 3종 — Fixed / Recursive / Markdown
# ─────────────────────────────────────────────────────
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

# (1) Fixed 512 — 가장 단순, 의미 단위 무시
fixed = [full_text[i:i+512] for i in range(0, len(full_text), 512)]

# (2) Recursive 512 / 64 — 구분자 우선순위 (\n\n → \n → 공백)
recursive = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=64
).split_text(full_text)

# (4) Markdown-aware — # 헤딩 계층으로 분할
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#","ch"),("##","sec"),("###","sub")]
)
md_chunks = [d.page_content for d in md_splitter.split_text(full_text)]

print(f"Fixed   : {len(fixed)} chunks")
print(f"Recursive: {len(recursive)} chunks")
print(f"Markdown : {len(md_chunks)} chunks")

Fixed   : 1 chunks
Recursive: 1 chunks
Markdown : 2 chunks


### 2-3-2. Semantic 청킹 — 임베딩 유사도 경계

LangChain Experimental의 `SemanticChunker`가 인접 문장의 임베딩 유사도가 급격히 떨어지는 지점에서 자른다. 인덱싱 비용이 약간 들지만 의미적 응집도가 가장 높다.

In [9]:
# ─────────────────────────────────────────────────────
# (3) Semantic chunking — 인접 문장 임베딩 유사도 기반
# ─────────────────────────────────────────────────────
from langchain_experimental.text_splitter import SemanticChunker

try:
    sem_splitter = SemanticChunker(
        OpenAIEmbeddings(model="text-embedding-3-small"),
        breakpoint_threshold_type="percentile",  # 상위 N% 변화점에서 자르기
    )
    semantic = sem_splitter.split_text(full_text)
    print(f"Semantic: {len(semantic)} chunks")
except Exception as e:
    semantic = recursive  # 대안
    print(f"Semantic 스킵 (langchain_experimental 미설치) — Recursive로 대체")

Semantic: 2 chunks


### 2-3-3. Contextual Retrieval — Anthropic의 검색 실패율 67% 감소 기법

각 청크 앞에 LLM이 작성한 "문서 전체 맥락 요약"을 prepend한다. 비용은 청크 수 × LLM 호출이지만, 리랭킹과 결합 시 검색 실패율이 최대 67% 감소한다 (Anthropic, 2024).

In [10]:
# ─────────────────────────────────────────────────────
# (7) Contextual Retrieval (Anthropic 2024 — OpenAI로 대체 구현)
# ─────────────────────────────────────────────────────
def contextual_chunk(chunk: str, full_doc: str) -> str:
    """각 청크 앞에 LLM이 작성한 50자 컨텍스트를 prepend"""
    r = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content":
            f"다음 청크가 문서에서 어떤 맥락에 속하는지 50자 이내로 작성하라.\n\n"
            f"[전체 문서 앞부분]\n{full_doc[:1500]}\n\n"
            f"[청크]\n{chunk}"}],
        max_completion_tokens=80,
    )
    ctx = r.choices[0].message.content.strip()
    # prepend 형식 — 검색·임베딩 시 컨텍스트가 청크의 일부로 동작
    return f"[맥락: {ctx}]\n{chunk}"

# 데모용으로 첫 3개 청크에만 적용 (비용 절감)
contextual = [contextual_chunk(c, full_text) for c in recursive[:3]]
print("--- contextual chunk 예시 ---")
print(contextual[0])

--- contextual chunk 예시 ---
[맥락: FY2025 연례보고서의 매출 및 리스크 요약 부분]
## **FY2025 Annual Report — Sample Corp** 

## **Chapter 1. Revenue** 

## **1.1 Q1 Results** 

|**Segment**|**Q1 Revenue (M$)**|**YoY %**|
|---|---|---|
|iPhone|69,702|+5.5%|
|Mac|7,744|+1.6%|
|Services|26,375|+11.5%|



## **Chapter 2. Risk Factors** 

Supply chain disruption is the primary risk identified.


### 2-3-4. Hit@1 정량 비교 — 청킹 전략별 검색 정확도

골드 키워드가 Top-1 청크에 포함되는지를 측정한다. 더 정밀한 평가는 RAGAS context_precision이지만, 본 데모는 간이 신호로 충분하다.

In [12]:
# ─────────────────────────────────────────────────────
# 청킹 전략별 Hit@1 측정
# ─────────────────────────────────────────────────────
def hit_rate(chunks, queries, gold_keywords):
    """쿼리별로 top-1 청크가 정답 키워드를 포함하는 비율"""
    # 빈 청크 제거 (semantic chunking에서 종종 발생)
    chunks = [c for c in chunks if c and c.strip()]
    if not chunks:
        return 0.0

    vecs = embed([c[:1024] for c in chunks])
    qv   = embed(queries)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    qn    = np.linalg.norm(qv,   axis=1, keepdims=True)
    # 0 division 방어 (eps 추가)
    sims = (qv / np.maximum(qn, 1e-12)) @ (vecs / np.maximum(norms, 1e-12)).T
    top1 = sims.argmax(axis=1)
    hits = sum(1 for i, k in enumerate(gold_keywords)
               if k in chunks[top1[i]])
    return hits / len(queries)

gold_q = ["iPhone Q1 매출", "Mac 매출", "Services 매출",
          "Risk Factor", "FY2025"]
gold_k = ["iPhone",        "Mac",     "Services",
          "Supply chain",   "FY2025"]

scores = {
    "Fixed":     hit_rate(fixed[:30],     gold_q, gold_k),
    "Recursive": hit_rate(recursive[:30], gold_q, gold_k),
    "Markdown":  hit_rate(md_chunks,      gold_q, gold_k) if md_chunks else 0,
    "Semantic":  hit_rate(semantic[:30],  gold_q, gold_k),
}
pd.DataFrame(scores.items(),
             columns=["전략", "Hit@1"]).sort_values("Hit@1", ascending=False)

,전략,Hit@1
0,Fixed,1.0
1,Recursive,1.0
3,Semantic,1.0
2,Markdown,0.8


---
## §2-4 프로덕션 파이프라인 — Qdrant + BM25 + RRF + Cohere Rerank

본격적인 프로덕션 등급 RAG를 구성한다. **Hybrid Search(BM25 + Dense) → RRF 결합 → Cohere Rerank v3.5**의 3단계 누적 효과를 측정한다.

이 파이프라인은 Chapter 7(최종 비교)의 VectorRAG 백엔드 baseline이 된다.

### 2-4-1. Qdrant 인메모리 컬렉션 + 벡터 업로드

Qdrant는 Rust 기반으로 20ms p95 지연을 자랑한다. `:memory:` 모드로 외부 서버 없이 시작할 수 있다.

In [13]:
# ─────────────────────────────────────────────────────
# §2-4: Qdrant 인메모리 클라이언트 + 컬렉션 생성
# ─────────────────────────────────────────────────────
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue,
)

qc = QdrantClient(":memory:")     # 외부 서버 없이 시작
qc.recreate_collection(
    collection_name="ch02_prod",
    vectors_config=VectorParams(
        size=1536,                 # text-embedding-3-small 차원
        distance=Distance.COSINE,  # 코사인 유사도
    ),
)

# 청크 임베딩 후 일괄 업로드
chunk_vecs = embed(recursive[:50])
points = [
    PointStruct(id=i, vector=v.tolist(),
                payload={"text": recursive[i], "chunk_id": i})
    for i, v in enumerate(chunk_vecs)
]
qc.upsert(collection_name="ch02_prod", points=points)
print(f"Qdrant 컬렉션 업로드 완료: {len(points)} points")

Qdrant 컬렉션 업로드 완료: 1 points


/var/folders/v9/46y9d8bn1lxgjt7g439hsf8c0000gn/T/ipykernel_48497/838443992.py:10: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qc.recreate_collection(


### 2-4-2. BM25 sparse 검색기 + Dense·Sparse 두 결과 비교

BM25는 정확한 단어·고유명사·코드 식별자에 강하다. Dense는 의미가 유사하지만 단어가 다른 문서를 잘 찾는다. 두 결과의 차이가 RRF 결합의 근거가 된다.

In [14]:
# ─────────────────────────────────────────────────────
# BM25 (sparse) 검색기
# ─────────────────────────────────────────────────────
from rank_bm25 import BM25Okapi

tokenized = [c.lower().split() for c in recursive[:50]]
bm25 = BM25Okapi(tokenized)

def dense_search(query, k=10):
    """Dense (임베딩) 검색"""
    qv = embed([query])[0].tolist()
    res = qc.search(collection_name="ch02_prod", query_vector=qv, limit=k)
    return [(r.payload["chunk_id"], r.score) for r in res]

def sparse_search(query, k=10):
    """BM25 (키워드) 검색"""
    scores = bm25.get_scores(query.lower().split())
    idx = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in idx]

q = "iPhone Q1 revenue"
print("Dense top-5:", dense_search(q, 5))
print("BM25  top-5:", sparse_search(q, 5))

Dense top-5: [(0, 0.5196673960731674)]
BM25  top-5: [(0, -0.5493061443340549)]


/var/folders/v9/46y9d8bn1lxgjt7g439hsf8c0000gn/T/ipykernel_48497/1515011371.py:12: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  res = qc.search(collection_name="ch02_prod", query_vector=qv, limit=k)


### 2-4-3. RRF (Reciprocal Rank Fusion) 결합

두 검색 결과의 순위를 1/(k + rank) 가중치로 합산한다. k=60이 표준값이다. 동일 청크가 두 리스트에 모두 등장하면 점수가 누적되어 상위로 올라간다.

In [15]:
# ─────────────────────────────────────────────────────
# RRF 결합 — Dense + Sparse 두 순위 리스트 합치기
# ─────────────────────────────────────────────────────
def rrf_combine(*ranked_lists, k=60, top_n=10):
    """Reciprocal Rank Fusion — score = Σ 1/(k + rank)"""
    scores = {}
    for ranked in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
    fused = sorted(scores.items(), key=lambda x: -x[1])[:top_n]
    return fused

hybrid = rrf_combine(dense_search(q, 10), sparse_search(q, 10))
print("Hybrid RRF top-10:")
for doc_id, score in hybrid:
    print(f"  chunk_id={doc_id}  rrf_score={score:.4f}")

Hybrid RRF top-10:
  chunk_id=0  rrf_score=0.0333


/var/folders/v9/46y9d8bn1lxgjt7g439hsf8c0000gn/T/ipykernel_48497/1515011371.py:12: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  res = qc.search(collection_name="ch02_prod", query_vector=qv, limit=k)


### 2-4-4. Cohere Rerank v3.5 — Cross-encoder 정밀 재정렬

RRF로 정렬한 Top-10을 Cohere의 cross-encoder가 쿼리·문서 동시 입력 방식으로 정밀 재정렬한다. 1k 검색당 $2, 평균 ~600ms 지연이다.

In [16]:
# ─────────────────────────────────────────────────────
# Cohere Rerank v3.5 — cross-encoder 정밀 재정렬
# ─────────────────────────────────────────────────────
if os.environ.get("COHERE_API_KEY"):
    import cohere
    co = cohere.Client(os.environ["COHERE_API_KEY"])
    docs = [recursive[i] for i, _ in hybrid]
    rr = co.rerank(
        model="rerank-v3.5",          # 2026-05 표준
        query=q,
        documents=docs,
        top_n=5,
    )
    print("Cohere Rerank top-5:")
    for r in rr.results:
        print(f"  idx={r.index} relevance={r.relevance_score:.3f}")
else:
    print("⚠️ COHERE_API_KEY 미설정 — Rerank 단계는 스킵")
    print("👉 https://cohere.com에서 무료 키 발급 후 환경 변수 설정")

Cohere Rerank top-5:
  idx=0 relevance=0.852


---
## §2-5 VectorRAG가 실패하는 5가지 시나리오 재현

이론 그림 2.6의 5가지 실패 모드를 §2-1의 베이스라인으로 직접 재현한다. **이 다섯 실패가 Chapter 3~6의 VectorlessRAG 계열이 등장한 이유다.**

In [17]:
# ─────────────────────────────────────────────────────
# §2-5: 5가지 실패 시나리오 질의 실행
# ─────────────────────────────────────────────────────
failure_queries = {
    "1. 다중 엔티티":  "iPhone과 Mac과 Services 모두에서 매출이 증가한 분기는?",
    "2. 표 셀 단절":   "FY2025 Q1 Services 부문 정확한 매출 수치는 얼마인가? (단위 포함)",
    "3. 교차 참조":    "Chapter 2 Risk Factors에서 Chapter 1의 어느 부문이 언급되는가?",
    "4. 시점 비교":    "FY2024 대비 FY2025 Q1 iPhone 매출 증감액은 얼마인가?",
    "5. 반의어/부정":  "매출이 감소한 사업부문은 무엇인가?",
}

print("=== 베이스라인 VectorRAG 답변 ===")
for name, q in failure_queries.items():
    print(f"\n[{name}]")
    print(f"Q: {q}")
    try:
        a = rag.invoke(q)
        print(f"A: {a[:200]}")
    except Exception as e:
        print(f"A: <오류> {e}")

=== 베이스라인 VectorRAG 답변 ===

[1. 다중 엔티티]
Q: iPhone과 Mac과 Services 모두에서 매출이 증가한 분기는?


Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1


A: 문서에 명시되지 않았다.

[2. 표 셀 단절]
Q: FY2025 Q1 Services 부문 정확한 매출 수치는 얼마인가? (단위 포함)


Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1


A: FY2025 Q1 Services 부문의 정확한 매출 수치는 **26,375 M$**입니다.

[3. 교차 참조]
Q: Chapter 2 Risk Factors에서 Chapter 1의 어느 부문이 언급되는가?


Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1


A: 문서에 명시되지 않았다.

[4. 시점 비교]
Q: FY2024 대비 FY2025 Q1 iPhone 매출 증감액은 얼마인가?


Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1


A: 문서에 명시된 FY2025 Q1 iPhone 매출은 69,702M$이며, FY2024 수치는 문서에 명시되지 않았다. 따라서 FY2024 대비 증감액은 문서에 명시되지 않았다.

[5. 반의어/부정]
Q: 매출이 감소한 사업부문은 무엇인가?
A: 문서에 명시되지 않았다.


**관찰 패턴**:
- **1. 다중 엔티티**: Top-k 청크에 3개 부문 정보가 동시에 들어오지 못해 답을 합성하지 못한다.
- **2. 표 셀 단절**: 청킹 경계가 표 헤더와 데이터를 분리하여 "Services 매출"이 다른 행과 혼동될 수 있다.
- **3. 교차 참조**: 두 장에 흩어진 정보가 의미적으로 멀어 동시에 검색되지 않는다.
- **4. 시점 비교**: FY2024와 FY2025 청크의 임베딩이 매우 유사해 둘 중 하나만 가져온다.
- **5. 반의어/부정**: "매출 증가"와 "매출 감소"의 임베딩이 유사하여 잘못된 청크를 반환한다.

**다음 챕터 — Chapter 3**에서는 이 5가지를 정면 돌파하는 새 패러다임(VectorlessRAG)을 이론과 PageIndex 시스템으로 소개한다.